In [1]:
# ==========================================
# # 1. SETUP DE AMBIENTE (Importações de bibliotecas)
# ==========================================

import pandas as pd  # Manipulação de tabelas e dados estruturados
import numpy as np # Operações matemáticas e vetoriais
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix # Métricas de Avaliação
from sklearn.model_selection import train_test_split # Divisão da base

# NOVAS IMPORTAÇÕES PARA MISTURAR TEXTO E NÚMERO:
from sklearn.preprocessing import StandardScaler, OneHotEncoder # Padronização e conversão de colunas de texto
from sklearn.compose import ColumnTransformer # Aplica transformações por colunas específicas

from tensorflow import keras # Framework de Deep Learning       # type: ignore
from tensorflow.keras import layers # Camadas da rede neural    # type: ignore

In [2]:
# ==========================================
# # 2. CARGA E EXPLORAÇÃO (Leitura dos dados, .info(), gráficos)
# ==========================================

df = pd.read_csv('vendas.csv') # Lê a entrada
df.head() # Exibe as primeiras 5 linhas
print(df.info()) # Informações sobre tipo de dados, linhas e memória
print(df.describe()) # Extrai e exibe estatísticas descritivas
print(df.isnull().sum()) # Mapeia dados nulos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_cliente      200 non-null    int64  
 1   nome            200 non-null    object 
 2   idade           198 non-null    float64
 3   gasto_total     198 non-null    float64
 4   meses_contrato  199 non-null    float64
 5   plano           198 non-null    object 
 6   genero          198 non-null    object 
 7   cancelou        200 non-null    object 
dtypes: float64(3), int64(1), object(4)
memory usage: 12.6+ KB
None
        id_cliente       idade  gasto_total  meses_contrato
count   200.000000  198.000000   198.000000      199.000000
mean   1100.500000   37.893939  2677.777778       21.000000
std      57.879185   12.294766  2303.018569       12.069663
min    1001.000000   19.000000   150.000000        3.000000
25%    1050.750000   28.250000   600.000000       12.000000
50%    1100.500000

In [3]:
# ==========================================
# # 3. PRÉ-PROCESSAMENTO (Limpeza de nulos e Engenharia de Atributos)
# ==========================================

# Limpeza padrão de nulos das colunas numéricas
df['idade'] = df['idade'].fillna(df['idade'].median()) # Substitui valores de idades nulos pela mediana
df['gasto_total'] = df['gasto_total'].fillna(0) # Substitui valores de gasto_mensal por 0 levando em consideração o tempo de contrato neste caso específico
df['meses_contrato'] = df['meses_contrato'].fillna(df['meses_contrato'].median()) # Substitui valores de meses_contrato pela mediana

# Limpeza de nulos das NOVAS colunas de texto usando a moda (o valor que mais se repete)
df['plano'] = df['plano'].fillna(df['plano'].mode()[0])
df['genero'] = df['genero'].fillna(df['genero'].mode()[0])

# Cria uma nova coluna chamada gasto_mensal para ajudar a entender o comportamento do cliente
# Insere a nova coluna apenas se ela ainda não existir
if 'gasto_mensal' not in df.columns:
    df.insert(5, 'gasto_mensal', df['gasto_total'] / df['meses_contrato'])
df

,id_cliente,nome,idade,gasto_total,meses_contrato,gasto_mensal,plano,genero,cancelou
0,1001,Carlos Silva,34.0,1500.0,12.0,125.000000,Premium,Masculino,Nao
1,1002,Ana Souza,22.0,300.0,6.0,50.000000,Basic,Feminino,Sim
2,1003,Roberto Lima,45.0,4500.0,24.0,187.500000,Premium,Masculino,Nao
3,1004,Juliana Costa,36.0,1200.0,12.0,100.000000,Standard,Feminino,Nao
4,1005,Marcos Oliveira,50.0,0.0,36.0,0.000000,Premium,Masculino,Nao
...,...,...,...,...,...,...,...,...,...
195,1196,Carlos Silva,60.0,4500.0,36.0,125.000000,Standard,Masculino,Nao
196,1197,Roberto Souza,25.0,600.0,12.0,50.000000,Basic,Masculino,Sim
197,1198,Marcos Lima,30.0,1200.0,18.0,66.666667,Standard,Masculino,Nao
198,1199,Ricardo Costa,40.0,2400.0,24.0,100.000000,Premium,Masculino,Nao


In [4]:
# ==========================================
# 4. PREPARAÇÃO PARA O MODELO (Mapeamento de colunas e transformação,Separa X e y, divide treino e teste, padronização de dados númericos)
# ==========================================

# Separação de X preditores e y classe alvo
X = df.drop(columns=['id_cliente', 'nome', 'cancelou'])
y = df['cancelou']
y = y.map({'Nao': 0, 'Sim': 1}) ## Converte a classe categórica em numérica: 0 para Não e 1 para Sim

# Divisão de Treino e Teste antes de qualquer transformação (Evita vazamento)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Separa 80% para treino e 20% para teste. random_state garante que pode ser reproduzido de maneira semelhante

# Mapeamos quais colunas são numéricas e quais são de texto (essencial para o ColumnTransformer)
colunas_numericas = ['idade', 'gasto_total', 'meses_contrato', 'gasto_mensal']
colunas_categoricas = ['plano', 'genero']

# Criamos o transformador que aplica a ferramenta certa na coluna certa
preprocessador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), colunas_numericas),          # Aplica escala nos números
    ('cat', OneHotEncoder(drop='first'), colunas_categoricas) # Transforma textos em colunas 0 e 1
])

# Aplica o pré-processador nos dados (O treino aprende e transforma; o teste apenas transforma)
X_train_scaled = preprocessador.fit_transform(X_train)
X_test_scaled = preprocessador.transform(X_test) # CORRIGIDO: Apenas .transform() aqui!

In [5]:
# ==========================================
# # 5. TREINAMENTO DO MODELO
# ==========================================

# Inicializa um modelo Sequencial e guarda na varivel model
model = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)), # Camada de Processamento Inicial(com 32 neurônios e ativação Relu introduz não-linearidade)e que também define o formato dos dados de entrada (input_shape) usando o número de colunas de X
    layers.Dropout(0.2), # Desativa aleatoriamente 20% dos neurônios para evitar overfitting
    layers.Dense(16, activation='relu'), # Segunda camada oculta com 16 neurônios e ativação relu para refinar os padrões aprendidos
    layers.Dense(1, activation='sigmoid') # Camada de saída com 1 neurônio e ativação Sigmoid para retornar uma probabilidade entre 0 e 1 (classificação binária)
])

# Compila o modelo definindo o algoritmo de otimização adam, a função de perda binária binary_crossentropy e a métrica de avaliação accuracy
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Treina a rede neural salvando o histórico na variável treino
# Passa os dados de treino padronizados (X_train_scaled e y_train)
# Executa 25 épocas (ciclos completos de leitura da base)
# Processa os dados em lotes (batch_size) de 32 em 32 linhas
# Separa 10% dos dados (validation_split) para validação em tempo real
# Mostra a barra de progresso do aprendizado na tela (verbose=1)
treino = model.fit(X_train_scaled, y_train, epochs=25, batch_size=32, validation_split=0.1, verbose=1)

Epoch 1/25


c:\Users\adess\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.5694 - loss: 0.6984 - val_accuracy: 0.7500 - val_loss: 0.6428
Epoch 2/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6319 - loss: 0.6670 - val_accuracy: 0.7500 - val_loss: 0.6149
Epoch 3/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7014 - loss: 0.6260 - val_accuracy: 0.7500 - val_loss: 0.5870
Epoch 4/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7431 - loss: 0.5972 - val_accuracy: 0.7500 - val_loss: 0.5646
Epoch 5/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7847 - loss: 0.5744 - val_accuracy: 0.7500 - val_loss: 0.5373
Epoch 6/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8472 - loss: 0.5412 - val_accuracy: 0.8125 - val_loss: 0.5124
Epoch 7/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8819 - loss: 0.5035 - val_accuracy: 0.8125 - val_loss: 0.4832
Epoch 8/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8681 - loss: 0.4740 - val_accuracy: 0.8125 - val_loss: 0.4549
Epoch 9/25


In [6]:
# =======================================================
# 6. AVALIAÇÃO DE PERFORMANCE
# =======================================================

# 1. Avalia o modelo diretamente com os dados de teste (Retorna a perda e a acurácia global)
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

# 2. Gera as probabilidades de cancelamento (valores entre 0 e 1) para a base de teste
y_pred_probs = model.predict(X_test_scaled, verbose=0)

# 3. Converte as probabilidades em classes binárias: vira 1 se for maior que 0.5, senão vira 0
y_pred = (y_pred_probs > 0.5).astype(int)

# 4. Calcula as métricas de classificação cruzando as respostas reais (y_test) com as previsões (y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# 5. Exibe o resumo de todas as métricas formatadas com 4 casas decimais
print("--- Resumo das Métricas de Performance ---")
print(f"Acurácia:  {accuracy:.4f} -> Porcentagem geral de acertos do modelo.")
print(f"Precision: {precision:.4f} -> Dos que o modelo previu que iam cancelar, quantos realmente cancelaram.")
print(f"Recall:    {recall:.4f} -> De todos os que realmente cancelaram, quantos o modelo conseguiu encontrar.")
print(f"F1-Score:  {f1:.4f} -> Equilíbrio (média harmônica) entre Precision e Recall.")

# 6. Gera e estrutura a Matriz de Confusão em um DataFrame para leitura clara
matriz = confusion_matrix(y_test, y_pred)
matriz_df = pd.DataFrame(
    matriz, 
    columns=['Previu: Ficou (0)', 'Previu: Cancelou (1)'],
    index=['Real: Ficou (0)', 'Real: Cancelou (1)']
)

print("\n--- Matriz de Confusão ---")
display(matriz_df) # Usa display() em vez de print para tabelas ficarem lindas no Jupyter/Colab

--- Resumo das Métricas de Performance ---
Acurácia:  0.9500 -> Porcentagem geral de acertos do modelo.
Precision: 1.0000 -> Dos que o modelo previu que iam cancelar, quantos realmente cancelaram.
Recall:    0.8571 -> De todos os que realmente cancelaram, quantos o modelo conseguiu encontrar.
F1-Score:  0.9231 -> Equilíbrio (média harmônica) entre Precision e Recall.

--- Matriz de Confusão ---


,Previu: Ficou (0),Previu: Cancelou (1)
Real: Ficou (0),26,0
Real: Cancelou (1),2,12
